In [ ]:
"""
Conformal Prediction Uncertainty Quantification for Ensemble Regressors.

This script applies Cross-Validation Conformal Prediction to top-performing 
ensemble models (GradientBoosting, RandomForest, XGBoost) for microstructure-based 
hardness prediction. It calculates 95% coverage prediction intervals and exports 
comparative performance logs and publication-quality error-bar plots.
"""

import os
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

fontsize = 35

# =============================================================================
# 1. Initialize Output Directory & Load Cleaned Dataset
# =============================================================================

output_dir = r"./data/figure_conformal"
os.makedirs(output_dir, exist_ok=True)

df_filtered = pd.read_csv(r"./data/d_hv_with_corr_features.csv")

target_column = "HV"
drop_cols = ["FILE_NAME", target_column]
feature_cols = [c for c in df_filtered.columns if c not in drop_cols]

# Extract feature and target matrices
X = df_filtered[feature_cols].values
y = df_filtered[target_column].values

# Standardization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# =============================================================================
# 2. Define Ensemble Regression Models & Cross-Validation
# =============================================================================

models = {
    "GradientBoosting": GradientBoostingRegressor(n_estimators=200, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
}


kf = KFold(n_splits=5, shuffle=True, random_state=42)

# =============================================================================
# 3. Model Evaluation and Conformal Prediction Pipeline
# =============================================================================

results = []
conformal_results = []

# Conformal Prediction setup (1 - alpha = 0.95 -> 95% Coverage Interval)
alpha = 0.05
n_samples = len(y)
q_level = np.ceil((n_samples + 1) * (1 - alpha)) / n_samples
q_level = min(1.0, q_level)

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 7))

print(f"[INFO] Initiating Conformal Prediction analysis for {len(models)} ensemble models...\n")
print(f"{'Model':<20} | {'R²':>8} | {'RMSE':>8} | {'MAE':>8} | {'95% CP Margin (±HV)':>22} | {'Coverage':>10}")
print("-" * 82)

for i, (name, model) in enumerate(models.items()):

    # Cross-validation prediction profile
    y_pred = cross_val_predict(model, X_scaled, y, cv=kf)

    # Compute regression validation metrics
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    mae = mean_absolute_error(y, y_pred)

    # Calculate absolute residuals
    abs_error = np.abs(y - y_pred)

    # Conformal prediction interval margin (q_hat)
    q_hat = np.quantile(abs_error, q_level)

    # Compute bounds and empirical coverage
    y_lower = y_pred - q_hat
    y_upper = y_pred + q_hat
    coverage = np.mean((y >= y_lower) & (y <= y_upper)) * 100

    results.append({"Model": name, "R2": r2, "RMSE": rmse, "MAE": mae})
    conformal_results.append({
        "Model": name,
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "CP_Margin_HV": q_hat,
        "Empirical_Coverage_%": coverage,
    })

    print(f"{name:<20} | {r2:>8.3f} | {rmse:>8.3f} | {mae:>8.3f} | {q_hat:>22.3f} | {coverage:>9.1f}%")

    # Determine aligned data range for identical scaling
    min_val = min(y.min(), y_pred.min())
    max_val = max(y.max(), y_pred.max())

    # -------------------------------------------------------------------------
    # Save Individual Prediction Figure with Error Bars (Standalone Analysis)
    # -------------------------------------------------------------------------
    fig_single, ax_single = plt.subplots(figsize=(10, 10))

    ax_single.errorbar(
        y,
        y_pred,
        yerr=q_hat,
        fmt="o",
        color="#2b5c8f",
        ecolor="#9ebcda",
        elinewidth=1,
        capsize=2,
        alpha=0.6,
        label=f"Pred ± {q_hat:.2f} (95% CP)",
    )
    ax_single.plot([min_val, max_val], [min_val, max_val], "r--", lw=2, label=f"$R^2 = {r2:.3f}$")

    ax_single.set_xlabel("Actual HV", fontsize=fontsize)
    ax_single.set_ylabel("Predicted HV", fontsize=fontsize)
    ax_single.tick_params(axis="both", labelsize=fontsize)
    ax_single.legend(loc="upper left", fontsize=fontsize, frameon=True)
    ax_single.grid(True, linestyle=":", alpha=0.6)

    fig_single.tight_layout()
    single_save_path = os.path.join(output_dir, f"{name.lower()}_conformal_actual_vs_predicted.png")
    fig_single.savefig(single_save_path, dpi=300, bbox_inches="tight")
    plt.close(fig_single)

    # -------------------------------------------------------------------------
    # Populate Benchmark Grid Figure
    # -------------------------------------------------------------------------
    ax = axes[i]
    ax.errorbar(
        y,
        y_pred,
        yerr=q_hat,
        fmt="o",
        color="#2b5c8f",
        ecolor="#9ebcda",
        elinewidth=1,
        capsize=2,
        alpha=0.5,
        label=f"{name}: ±{q_hat:.2f}",
    )
    ax.plot([min_val, max_val], [min_val, max_val], "r--", lw=2, label=f"$R^2 = {r2:.3f}$")
    ax.set_xlabel("Actual HV", fontsize=fontsize)
    ax.set_ylabel("Predicted HV", fontsize=fontsize)
    ax.tick_params(axis="both", labelsize=fontsize)
    ax.legend(loc="upper left", fontsize=fontsize, frameon=True)
    ax.grid(True, linestyle=":", alpha=0.6)

grid_save_path = os.path.join(output_dir, "ensemble_conformal_comparison_all.png")
fig.tight_layout()
fig.savefig(grid_save_path, dpi=300, bbox_inches="tight")
plt.close(fig)

# =============================================================================
# 4. Export Conformal Prediction Results
# =============================================================================

df_conformal = pd.DataFrame(conformal_results)
df_conformal = df_conformal.sort_values("CP_Margin_HV", ascending=True).reset_index(drop=True)

print("\n[INFO] Conformal Prediction ranking (sorted by narrowest uncertainty margin)")
print(df_conformal.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

export_csv_path = r"./data/f_ensemble_conformal_results.csv"
df_conformal.to_csv(export_csv_path, index=False)

print(f"\n[INFO] Conformal Prediction evaluation completed successfully.")
print(f"[INFO] Performance logs saved to: {export_csv_path}")
print(f"[INFO] Summary figures exported to: {output_dir}")


# =============================================================================
# 5. Final Model Save
# =============================================================================

final_gb_model = GradientBoostingRegressor(
    n_estimators=200,
    random_state=42
)

final_gb_model.fit(X_scaled, y)

model_save_path = os.path.join(output_dir, "GradientBoosting_HV_prediction_model.pkl")
joblib.dump(
    {
        "model": final_gb_model,
        "scaler": scaler,
        "feature_names": feature_cols
    },
    model_save_path
)

model_size_MB = os.path.getsize(model_save_path) / (1024 ** 2)

print(f"\n[INFO] Final Gradient Boosting model saved: {model_save_path}")
print(f"[INFO] Saved model size: {model_size_MB:.3f} MB")